# 第 6 章の講義と実習　表の結合

最初に，二つの府中市で年ごとの行を正しく組にする方法を確かめます。後半の約 20 分は別の市を選び，どの行を対応させたかを根拠とともに報告します。

## 準備

那珂川市（福岡県）か丹波篠山市（兵庫県）を一つ選び，2015 年から 2020 年に人口が何人増減したかを調べます。市名やコードが二つの年で異なる場合があるため，人口を引く前に統計局の変更記録で同じ地域の行を探します。

Colab では「ドライブにコピーを保存」で自分用のノートを作り，上から準備セルを実行します。CSV は公開 URL から読み込むため，アップロードは不要です。ローカルでは，実習 ZIP を展開してノートブックと CSV を同じフォルダーに置くと，手元のファイルを使えます。

最初のコードはデータが置いてあるフォルダーを探し，`code` を 5 桁の文字列として読みます。実行すると，2015 年，2020 年の順に行数が表示されます。

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display

file_2015 = "census-municipalities-2015.csv"
file_2020 = "census-municipalities-2020.csv"
data_folder = Path("data")
if not (data_folder / file_2015).is_file():
    data_folder = Path("../data")
if not (data_folder / file_2015).is_file():
    data_folder = Path(".")

path_2015 = data_folder / file_2015
path_2020 = data_folder / file_2020
if not path_2015.is_file():
    path_2015 = "https://raw.githubusercontent.com/sohei-takagi/python-ds-course-materials/main/data/" + file_2015
if not path_2020.is_file():
    path_2020 = "https://raw.githubusercontent.com/sohei-takagi/python-ds-course-materials/main/data/" + file_2020

code_type = {"code": "str"}
census_2015 = pd.read_csv(path_2015, dtype=code_type)
census_2020 = pd.read_csv(path_2020, dtype=code_type)
len(census_2015), len(census_2020)

## 講義中に実行するセル

見出しの順に実行し，市名だけで結合した組とコードで結合した組を比べます。各セルを動かす前に，どの地域の行が対応すると予想するか確かめてください。

### 1. 同じ名前の市を見分ける

同じ「府中市」が二つあるため，市名だけでは東京都の行を特定できません。2015 年の行だけを逆順に表示し，表の位置も手掛かりにならないことを確かめます。2020 年の東京都府中市に対応する 2015 年の行を，都道府県と 5 桁のコードから選んでください。

In [ ]:
lecture_fuchu_2020 = census_2020.loc[census_2020["municipality"] == "府中市"]
lecture_fuchu_2015 = census_2015.loc[census_2015["municipality"] == "府中市"]
lecture_fuchu_2015 = lecture_fuchu_2015.sort_values("code", ascending=False)
print("2020")
display(lecture_fuchu_2020)
print("2015 (descending code)")
display(lecture_fuchu_2015)

### 2. 市名とコードで結合した結果を比べる

結合で相手を探す列をキーと呼びます。`on` に市名を指定すると，2020 年の各府中市に 2015 年の相手が何行ずつ見つかるでしょうか。コードを指定した場合の行数も予想し，出力で比べます。正しい組を選べたら，同じ地域の 2020 年人口から 2015 年人口を引きます。

In [ ]:
lecture_name_pairs = lecture_fuchu_2020.merge(
    lecture_fuchu_2015, on="municipality", suffixes=("_2020", "_2015"),
)
lecture_name_pairs[["municipality", "prefecture_2020", "prefecture_2015"]]

In [ ]:
lecture_fuchu = lecture_fuchu_2020.merge(
    lecture_fuchu_2015, on="code", suffixes=("_2020", "_2015"),
    validate="one_to_one",
)
lecture_fuchu["population_change"] = (
    lecture_fuchu["population_2020"] - lecture_fuchu["population_2015"]
)
lecture_fuchu[["code", "prefecture_2020", "population_2015",
               "population_2020", "population_change"]]

### 3. 広島県府中市を旧区域に分ける

広島県府中市の人口差 −2,414 人のうち，どちらの旧区域で何人減ったのでしょうか。`current_code` は市全体を指すため二つの旧区域で共通ですが，`area_code` は区域を区別します。`area_code` で同じ区域の二時点を組にし，二つの差の合計が市全体の差と一致するか確かめます。

In [ ]:
lecture_area_file = "census-fuchu-areas.csv"
lecture_area_path = data_folder / lecture_area_file
if not lecture_area_path.is_file():
    lecture_area_path = (
        "https://raw.githubusercontent.com/sohei-takagi/python-ds-course-materials/main/data/"
        + lecture_area_file
    )
lecture_areas = pd.read_csv(
    lecture_area_path, dtype={"current_code": "str", "area_code": "str"},
)
lecture_areas

In [ ]:
lecture_areas_2020 = lecture_areas.loc[lecture_areas["year"] == 2020]
lecture_areas_2015 = lecture_areas.loc[lecture_areas["year"] == 2015]
lecture_area_pairs = lecture_areas_2020.merge(
    lecture_areas_2015, on="area_code", suffixes=("_2020", "_2015"),
    validate="one_to_one",
)
lecture_area_pairs["population_change"] = (
    lecture_area_pairs["population_2020"] - lecture_area_pairs["population_2015"]
)
display(lecture_area_pairs[["area_name_2020", "population_change"]])
print("Combined change:", lecture_area_pairs["population_change"].sum())

### 4. 2015 年の同じコードが見つからない行を探す

全国の行をコードで結合すると，2015 年の人口欄が空く市はあるでしょうか。`how="left"` は 2020 年の行を残し，`how="inner"` は両年でコードが一致した組だけを残します。行数と `_merge` の `left_only` を見て，空いた市の名前を特定してください。空欄を埋めるには，二つの調査の間のどんな変更を調べる必要がありますか。

In [ ]:
lecture_left = census_2020.merge(
    census_2015, on="code", how="left", suffixes=("_2020", "_2015"),
    validate="one_to_one", indicator=True,
)
lecture_inner = census_2020.merge(census_2015, on="code", how="inner")
print("Left:", len(lecture_left), "Inner:", len(lecture_inner))
lecture_left.loc[lecture_left["_merge"] == "left_only",
                 ["code", "municipality_2020", "population_2015", "_merge"]]

### 5. 年ごとの人口を一つの列に置く

`melt()` で年ごとの人口を一つの列に移しても，東京都府中市の 2015 年人口 260,274 人は消えません。変換後のどの行・どの列にあるかを探します。その 1 行は「一つの地域の一つの年の人口」です。市名で結合したときの 4 行は候補の組だったので，行数が同じでも意味が違うことを説明してください。

In [ ]:
lecture_fuchu_long = lecture_fuchu.melt(
    id_vars=["code", "prefecture_2020"],
    value_vars=["population_2015", "population_2020"],
    var_name="measurement",
    value_name="population",
)
lecture_fuchu_long.sort_values(["code", "measurement"])

## 自分で分析する実習（約 20 分）

### 1. 相手を探す（約 5 分）

那珂川市と丹波篠山市のどちらを調べるか決めます。下のセルの `selected_code` に，選んだ市の 2020 年のコードを入れて一行を表示してください。那珂川市は `40231`，丹波篠山市は `28221` です。この行の人口が，後で引き算する 2020 年側の値になります。

In [ ]:
selected_code = "40231"
selected_2020 = census_2020.loc[census_2020["code"] == selected_code].copy()
selected_2020

2020 年の名前だけを使っても，2015 年の表で対応する行が見つかるとは限りません。選んだ市を下の変更表で探し，旧名称と旧コード，何が変わったかを読みます。この表は，2015 年と 2020 年の国勢調査の間の変更をまとめたものです。[統計局の市区町村の変更情報・組替人口一覧](https://www.stat.go.jp/data/kokusei/2020/kekka/pdf/r04.pdf)

| 変更日 | 旧名称・コード | 内容 | 新名称・コード |
|---|---|---|---|
| 2016 年 10 月 10 日 | 富谷町 `04423` | 市制施行 | 富谷市 `04216` |
| 2018 年 10 月 1 日 | 那珂川町 `40305` | 市制施行 | 那珂川市 `40231` |
| 2019 年 5 月 1 日 | 篠山市 `28221` | 名称変更 | 丹波篠山市 `28221` |

**予想**　2020 年の `code` のまま 2015 年表を探すと相手が見つかりますか。変更表のどこからそう考えたか，一文で書いてください。

（ここに記入）

### 2. 二つの行を組にする（約 9 分）

本文の富谷市では，2020 年の元コードを残したまま，2015 年の富谷町を探すコードを `comparison_code` に入れました。見本を実行し，出力のどの列に新旧のコードが残るかを確かめます。自分の市でも，同じ方法で対応の根拠を後から読み返せるようにします。

In [ ]:
tomiya_2020 = census_2020.loc[census_2020["code"] == "04216"].copy()
tomiya_2020["comparison_code"] = "04423"

tomiya = tomiya_2020.merge(
    census_2015,
    left_on="comparison_code",
    right_on="code",
    how="left",
    suffixes=("_2020", "_2015"),
    validate="one_to_one",
    indicator=True,
)
tomiya[["code_2020", "municipality_2020", "code_2015", "municipality_2015", "_merge"]]

変更表で確かめた 2015 年側のコードを，次のセルの `comparison_code` に 5 桁の文字列で入れます。そのコードを持つ 2015 年の行を表示し，都道府県と当時の市町名が変更表と合うか確かめてください。コードが両年で同じ場合も，変更表を根拠にその値を入れます。

In [ ]:
comparison_code = ""

if comparison_code:
    display(census_2015.loc[census_2015["code"] == comparison_code])

二つの年を結合します。先ほど確認した旧コードを 2020 年の行に保存してから，その値と 2015 年表の `code` を照合します。下のセルの `#` を外し，`left_on` と `right_on` の `""` に，それぞれ照合する列名を入れてください。最初のコメント行は，比較用コードを新しい列に保存し，2020 年の元コードも残すための行です。

- 左の表 `selected_2020` は，元の `code` と，追加する `comparison_code` を持ちます。
- 右の表 `census_2015` では，2015 年のコードが `code` 列にあります。

自分が選んだ旧コードが左表のどの列に入るかを考え，右表のどの列からその値を探すか決めてください。

In [ ]:
selected_comparison = None

# selected_2020["comparison_code"] = comparison_code
# selected_comparison = selected_2020.merge(
#     census_2015,
#     left_on="",
#     right_on="",
#     how="left",
#     suffixes=("_2020", "_2015"),
#     validate="one_to_one",
#     indicator=True,
# )

次のセルは，結合で選んだ二つの行と人口差を表示します。`_merge` の `both` は指定したコードの一致を示しますが，その組が歴史的に同じ地域かは変更記録でも確かめてください。二つの人口が元の表の値と一致し，差が 2020 年 − 2015 年になっているか読みます。`left_only` のときは，2015 年表で探した 5 桁を見直します。

In [ ]:
if selected_comparison is None:
    print("結合を記入して実行すると，ここに結果を表示できます。")
else:
    selected_comparison["population_change"] = (
        selected_comparison["population_2020"] - selected_comparison["population_2015"]
    )
    display(selected_comparison[[
        "code_2020", "municipality_2020", "code_2015", "municipality_2015", "_merge",
    ]])
    display(selected_comparison[[
        "population_2015", "population_2020", "population_change",
    ]])

### 3. 人口の変化を報告する（約 6 分）

最初の問い「2015 年から 2020 年に何人増減したか」に答えます。読者が二つの人口と行の対応を確認できるよう，計算と変更記録を 3〜4 文で示してください。人口差だけから増減の理由は決められないため，理由を調べるのに必要な別の情報も挙げます。

1. 地域名，二つの調査時点それぞれの人口と，何人増えたか・減ったか。
2. どの二つのコードを照合したか。変更記録の何を根拠に同じ地域の組としたか。
3. 人口が変わった理由を調べるため，次にどの情報を確かめるか。その情報から何を知りたいか。

（ここに記入）

提出前にカーネルを再起動し，全セルを上から実行します。1 で選んだ市について差が表示され，その市の報告が残ったノートブックを保存してください。

#### もう一つの地域（任意）

もう一方でも，1 の地域選択からやり直せます。二つの地域で，名称が変わったこととコードが変わったことはどう違いますか。結果を保存するなら，一つ目の報告と区別して書きます。

#### 出典

総務省統計局の平成 27 年・令和 2 年国勢調査から，市町村と東京都特別区の総人口を教材用に抽出・加工しました。各年 10 月 1 日現在の人数です。[2015 年の原表](https://www.e-stat.go.jp/stat-search/files?cycle=0&layout=datalist&month=0&tclass1=000001089055&tclass2=000001089056&toukei=00200521&tstat=000001080615&year=20150) ・ [2020 年の原表](https://www.e-stat.go.jp/stat-search/files?cycle=0&layout=datalist&page=1&stat_infid=000032142410&tclass1=000001136466&toukei=00200521&tstat=000001136464)